# Patients - Silver Layer (dim_patient)

Ingestion incrémentale depuis Bronze avec déduplication sur `patient_id`. Utilise `foreachBatch` + `MERGE` pour upsert.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG    = "hospital"
SOURCE     = f"{CATALOG}.bronze.patient_raw"
TARGET     = f"{CATALOG}.silver.dim_patient"
QUARANTINE = f"{CATALOG}.silver.quarantine_patient"
CHECKPOINT = f"/Volumes/{CATALOG}/bronze/checkpoints/silver/dim_patient"

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET} (
  patient_id    STRING NOT NULL,
  first_name    STRING,
  last_name     STRING,
  gender        STRING,
  dob           DATE,
  city          STRING,
  row_hash      STRING,
  valid_from    TIMESTAMP,
  valid_to      TIMESTAMP,
  is_current    BOOLEAN,
  _source_file  STRING,
  _updated_at   TIMESTAMP
) COMMENT 'Dimension patient : SCD2 sur city, SCD1 sur les autres attributs'
""")

# Fonctions de masquage (recréées ici par sécurité)
spark.sql(f"""CREATE OR REPLACE FUNCTION {CATALOG}.silver.mask_pii_string(val STRING)
RETURN CASE WHEN is_account_group_member('data-engineers') THEN val ELSE '***' END""")
spark.sql(f"""CREATE OR REPLACE FUNCTION {CATALOG}.silver.mask_pii_date(val DATE)
RETURN CASE WHEN is_account_group_member('data-engineers') THEN val ELSE NULL END""")

# Application des masques sur les colonnes sensibles
spark.sql(f"ALTER TABLE {TARGET} ALTER COLUMN first_name SET MASK {CATALOG}.silver.mask_pii_string")
spark.sql(f"ALTER TABLE {TARGET} ALTER COLUMN last_name  SET MASK {CATALOG}.silver.mask_pii_string")
spark.sql(f"ALTER TABLE {TARGET} ALTER COLUMN dob        SET MASK {CATALOG}.silver.mask_pii_date")

In [0]:
def upsert_dim_patient(batch_df, batch_id):
    # 1. Typage et nettoyage
    typed = batch_df.select(
        F.upper(F.trim("patient_id")).alias("patient_id"),
        F.trim("first_name").alias("first_name"),
        F.trim("last_name").alias("last_name"),
        F.upper(F.trim("gender")).alias("gender"),
        F.expr("try_to_date(dob, 'yyyy-MM-dd')").alias("dob"),
        F.trim("city").alias("city"),
        "_rescued_data", "_source_file", "_ingested_at")

    # 2. Qualité
    reason = (F.when(F.col("patient_id").isNull() | (F.col("patient_id") == ""), "patient_id manquant")
               .when(F.col("dob").isNull(), "date de naissance invalide")
               .when(~F.col("gender").isin("M", "F"), "genre invalide")
               .when(F.col("_rescued_data").isNotNull(), "données inattendues"))
    checked = typed.withColumn("_reject_reason", reason)

    (checked.filter("_reject_reason IS NOT NULL")
        .withColumn("_rejected_at", F.current_timestamp())
        .write.mode("append")
        .option("txnAppId", "quarantine_patient").option("txnVersion", batch_id)
        .saveAsTable(QUARANTINE))

    # 3. Dernière version par patient + empreinte UNIQUEMENT sur l'attribut SCD2 (city)
    w = Window.partitionBy("patient_id").orderBy(F.col("_ingested_at").desc())
    latest = (checked.filter("_reject_reason IS NULL")
        .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1")
        .withColumn("row_hash", F.sha2(F.coalesce(F.col("city"), F.lit("")), 256)))

    # 4. SCD2 sur city (même mécanique que dim_hospital)
    current = (spark.table(TARGET).filter("is_current")
               .select("patient_id", F.col("row_hash").alias("current_hash")))
    joined = latest.join(current, "patient_id", "left")
    new_rows = (joined.filter("current_hash IS NULL")
                .withColumn("valid_from", F.lit("1900-01-01").cast("timestamp")))
    changed  = (joined.filter("current_hash IS NOT NULL AND current_hash <> row_hash")
                .withColumn("valid_from", F.col("_ingested_at")))

    cols = ["patient_id", "first_name", "last_name", "gender", "dob", "city",
            "row_hash", "valid_from", "_source_file"]
    staged = (new_rows.select(F.col("patient_id").alias("merge_key"), *cols)
        .unionByName(changed.select(F.col("patient_id").alias("merge_key"), *cols))
        .unionByName(changed.select(F.lit(None).cast("string").alias("merge_key"), *cols)))

    (DeltaTable.forName(spark, TARGET).alias("t")
        .merge(staged.alias("s"), "t.patient_id = s.merge_key AND t.is_current = true")
        .whenMatchedUpdate(
            condition="t.row_hash <> s.row_hash",
            set={"is_current": "false", "valid_to": "s.valid_from",
                 "_updated_at": "current_timestamp()"})
        .whenNotMatchedInsert(values={
            "patient_id": "s.patient_id", "first_name": "s.first_name",
            "last_name": "s.last_name", "gender": "s.gender", "dob": "s.dob",
            "city": "s.city", "row_hash": "s.row_hash", "valid_from": "s.valid_from",
            "valid_to": "CAST(NULL AS TIMESTAMP)", "is_current": "true",
            "_source_file": "s._source_file", "_updated_at": "current_timestamp()"})
        .execute())

    # 5. SCD1 sur les autres attributs : correction appliquée à TOUTES les versions du patient
    (DeltaTable.forName(spark, TARGET).alias("t")
        .merge(latest.alias("s"), "t.patient_id = s.patient_id")
        .whenMatchedUpdate(
            condition="""NOT (t.first_name <=> s.first_name AND t.last_name <=> s.last_name
                              AND t.gender <=> s.gender AND t.dob <=> s.dob)""",
            set={"first_name": "s.first_name", "last_name": "s.last_name",
                 "gender": "s.gender", "dob": "s.dob",
                 "_updated_at": "current_timestamp()"})
        .execute())

In [0]:
(spark.readStream.table(SOURCE)
    .writeStream.foreachBatch(upsert_dim_patient)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start().awaitTermination())

display(spark.table(TARGET).orderBy("patient_id", "valid_from"))